In [1]:
!pip install azure-ai-textanalytics azure-core

In [17]:
import os
os.environ["MicrosoftAIServiceEndpoint"] ="https://human-interaction-ms.cognitiveservices.azure.com/"
os.environ["MicrosoftAPIKey"] = "YOUR_KEY_HERE"

In [18]:
import os
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

endpoint = os.environ.get("MicrosoftAIServiceEndpoint")
api_key = os.environ.get("MicrosoftAPIKey")

text_analytics_client = None
if endpoint and api_key:
    text_analytics_client = TextAnalyticsClient(
        endpoint=endpoint, credential=AzureKeyCredential(api_key)
    )
else:
    print("Warning: Azure endpoint/key not found in environment variables.")

def call_ai_service(user_input):
    """
    Sends unmatched user input to Azure AI Language for sentiment analysis,
    then returns a response tailored to the detected sentiment. This is the
    extension point stubbed out in the original traditional_chatbot.ipynb.
    """
    if not text_analytics_client:
        return None

    try:
        result = text_analytics_client.analyze_sentiment([user_input])[0]
        sentiment = result.sentiment

        if sentiment == "positive":
            return "That sounds positive! I'm glad to hear it."
        elif sentiment == "negative":
            return "That sounds a little negative — I'm sorry if something's frustrating you."
        else:
            return "I'm not sure how to respond directly, but your tone seems neutral."
    except Exception as e:
        print("Azure sentiment analysis error:", e)
        return None

In [14]:
import re
import random

In [15]:
# Each rule: a compiled regex pattern paired with one or more possible responses.
# This structure is intentionally simple to extend later — a new rule is just
# one more (pattern, responses) tuple, and the fallback hook below is where
# a future AI service call would plug in.

rules = [
    (re.compile(r"\b(hi|hello|hey)\b", re.IGNORECASE),
        ["Hello! How can I help you today?", "Hi there! What can I do for you?"]),
    (re.compile(r"\bhow are you\b", re.IGNORECASE),
        ["I'm just a program, but I'm running smoothly! How about you?"]),
    (re.compile(r"\byour name\b", re.IGNORECASE),
        ["I'm a simple rule-based chatbot built for MSAI-631."]),
    (re.compile(r"\b(help|what can you do|capabilities)\b", re.IGNORECASE),
        ["I can greet you, tell you my name, chat a little, and respond to "
         "basic questions. Try asking 'what can you do', saying hello, or "
         "asking how I am."]),
    (re.compile(r"\b(bye|goodbye|exit|quit)\b", re.IGNORECASE),
        ["Goodbye! Have a great day."]),
    (re.compile(r"\bjoke\b", re.IGNORECASE),
        ["Why did the developer go broke? Because they used up all their cache."]),
]

def get_response(user_input):
    if not user_input or not user_input.strip():
        return "I didn't catch that — could you type something?"

    for pattern, responses in rules:
        if pattern.search(user_input):
            return random.choice(responses)

    ai_response = call_ai_service(user_input)
    if ai_response:
        return ai_response

    return ("I'm not sure how to respond to that yet. Try 'help' to see what I can do.")

In [16]:
print("Simple Chatbot (type 'bye' or 'quit' to exit)")
while True:
    user_input = input("You: ")
    response = get_response(user_input)
    print("Bot:", response)
    if re.search(r"\b(bye|goodbye|exit|quit)\b", user_input, re.IGNORECASE):
        break

Simple Chatbot (type 'bye' or 'quit' to exit)


You:  asdkfjaskdjf


Bot: I'm not sure how to respond directly, but your tone seems neutral.


KeyboardInterrupt: Interrupted by user